# Ablation: Omnigrok modular addition p=431, weight decay = 0

Контрольный запуск для проверки роли AdamW weight decay. Архитектура, split, mini-batch, learning rate и логирование совпадают с соответствующим WD>0 протоколом; изменён только `weight_decay=0.0`.

Это отдельный protocol name, поэтому TensorBoard и checkpoints не смешиваются с другими экспериментами.

## Что проверяет абляция

Сравниваются два режима:

- baseline с фиксированным ненулевым WD;
- этот запуск с `WD=0`.

Интерпретировать результат нужно по train/validation accuracy, loss, gap, weight norm, gradient/update norms, participation ratios, random projections и Fourier diagnostics. Если при WD=0 validation не генерализуется, это отрицательный контроль, а не ошибка эксперимента.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/grokking_prediction_original/2026-Project-202/code/Grokking/modular_addition_grokking_colab

In [ ]:
!pip install -q tensorboard

In [ ]:
from prime_sweep_omnigrok import Config, run_sweep

CONFIG = Config(
    output_root='/content/drive/MyDrive/grokking_prime_sweep',
    protocol_name='omnigrok_p431_v1_wd0_ablation',
    primes=(431,),
    seeds=(42,),

    train_fraction=0.30,
    normalize_train_examples_per_class=True,
    train_examples_per_class=34.0,
    max_sampled_pairs=500_000,

    d_model=128,
    d_mlp=512,
    num_heads=4,
    d_head=32,
    batch_size=512,
    batch_size_by_p={431: 512},
    model_dtype='float32',

    learning_rate=1e-3,
    betas=(0.9, 0.98),
    delayed_weight_decay=False,
    weight_decay=0.0,
    weight_decay_by_p={431: 0.0},

    max_steps=300,000,
    log_every=50,
    diagnostic_every=500,
    checkpoint_every=5_000,
    tensorboard_enabled=True,
    tensorboard_flush_secs=5,
    tensorboard_histogram_every=5_000,
    text_log_enabled=True,
    text_log_filename='training.log',
    text_log_every=1_000,

    monitor_train_pairs=2_048,
    monitor_val_pairs=2_048,
    eval_batch_size=2_048,
    projection_count=3,
    target_train_acc=0.99,
    target_val_acc=0.95,
    patience_logs=5,
    required_gap_steps=10_000,
    post_grok_steps=5_000,
    force_restart=True,
    skip_completed=False,
    device='auto',
    fused_adamw=True,
)
CONFIG

## Live TensorBoard

Запусти эту ячейку до обучения.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/grokking_prime_sweep/omnigrok_p431_v1_wd0_ablation --reload_interval 5

## Запуск обучения

Выполни следующую ячейку и дождись `summary` либо останови run вручную после достаточного числа шагов.

In [ ]:
summaries = run_sweep(CONFIG)
summaries

In [ ]:
from pathlib import Path
import pandas as pd

run_root = Path(CONFIG.output_root) / CONFIG.protocol_name / 'p_431' / 'seed_42'
print('run directory:', run_root)
for name in ('training_log.csv', 'training.log', 'summary.json', 'run_config.json'):
    path = run_root / name
    print(name, path.exists(), path)
if (run_root / 'training_log.csv').exists():
    display(pd.read_csv(run_root / 'training_log.csv').tail())

Для абляционного сравнения скачай `training_log.csv`, `summary.json` и `run_config.json`. Не объединяй WD=0 и WD>0 в один TensorBoard logdir без явного разделения тегов.